<a href="https://colab.research.google.com/github/QuintonPang/product-review-sentiment-ai/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import load_dataset, Dataset, concatenate_datasets
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from huggingface_hub import login
from google.colab import userdata
import os

login(token=userdata.get('HF_TOKEN'))

# ─────────────────────────────────────────
# CONFIG — toggle these to switch modes
# ─────────────────────────────────────────
MODE = "retrain"  # "initial" | "retrain"

INITIAL_MODEL   = "distilbert-base-uncased-finetuned-sst-2-english"
FINETUNED_MODEL = "quintonpyx/distilbert-amazon-polarity"
HUB_MODEL_ID    = "quintonpyx/distilbert-amazon-polarity"

TRAIN_SIZE = None   # None = full dataset, int = subset (e.g. 100_000)
TEST_SIZE  = None
FLAGGED_CSV = "flagged/log.csv"

# ─────────────────────────────────────────
# 1. Load base dataset
# ─────────────────────────────────────────
dataset = load_dataset("fancyzhx/amazon_polarity")

train_dataset = dataset["train"].shuffle(seed=42)
test_dataset  = dataset["test"].shuffle(seed=42)

if TRAIN_SIZE:
    train_dataset = train_dataset.select(range(TRAIN_SIZE))
if TEST_SIZE:
    test_dataset = test_dataset.select(range(TEST_SIZE))

def combine_text(example):
    example["text"] = example["title"] + ". " + example["content"]
    return example

train_dataset = train_dataset.map(combine_text).remove_columns(["title", "content"])
test_dataset  = test_dataset.map(combine_text).remove_columns(["title", "content"])

# ─────────────────────────────────────────
# 2. Mix in flagged corrections (retrain only)
# ─────────────────────────────────────────
if MODE == "retrain" and os.path.exists(FLAGGED_CSV):
    flagged_df = pd.read_csv(FLAGGED_CSV)[["input", "label"]]
    flagged_df = flagged_df.rename(columns={"input": "text"}).dropna()
    flagged_dataset = Dataset.from_pandas(flagged_df)
    print(f"Flagged samples: {len(flagged_dataset)}")
    train_dataset = concatenate_datasets([train_dataset, flagged_dataset]).shuffle(seed=42)

print(f"Total training samples: {len(train_dataset)}")

# ─────────────────────────────────────────
# 3. Load model
# ─────────────────────────────────────────
model_name = FINETUNED_MODEL if MODE == "retrain" else INITIAL_MODEL
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# ─────────────────────────────────────────
# 4. Tokenize
# ─────────────────────────────────────────
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=256)

train_dataset = train_dataset.map(tokenize_function, batched=True).remove_columns(["text"])
test_dataset  = test_dataset.map(tokenize_function, batched=True).remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ─────────────────────────────────────────
# 5. Metrics
# ─────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# ─────────────────────────────────────────
# 6. Training arguments (differ by mode)
# ─────────────────────────────────────────
is_retrain = MODE == "retrain"

training_args = TrainingArguments(
    output_dir=HUB_MODEL_ID,
    learning_rate=1e-5 if is_retrain else 2e-5,   # lower LR to avoid forgetting
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1 if is_retrain else 3,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=500 if is_retrain else 5000,
    save_strategy="steps",
    save_steps=500 if is_retrain else 5000,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=100 if is_retrain else 200,
    fp16=True,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="every_save",
)

# ─────────────────────────────────────────
# 7. Train
# ─────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# resume_from_checkpoint only makes sense for initial long runs
trainer.train(resume_from_checkpoint=is_retrain is False)

commit_msg = (
    "Retrained with flagged corrections" if is_retrain
    else f"Initial fine-tune, full dataset, 3 epochs, fp16"
)
trainer.push_to_hub(commit_message=commit_msg)
print(f"Done! Model live at: https://huggingface.co/quintonpyx/distilbert-amazon-polarity")

README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

amazon_polarity/train-00002-of-00004.par(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...olarity/model.safetensors:  48%|####7     |  128MB /  268MB            

  ...olarity/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

No files have been modified since last commit. Skipping to prevent empty commit.


Done! Model live at: https://huggingface.co/quintonpyx/distilbert-amazon-polarity
